# NSE all-index downloader (Google Colab)

Downloads the constituents of **every NSE index** (about 139) as one CSV per index, plus the master list
`NSE_ALL_INDICES_<timestamp>.csv`. Files go to `MyDrive/NSEDATA-ALL-INDEX/<YYYYMMDD>/`, the same place as before.

This version runs `code/nse_indices.py` (plain `requests`), so **no Chrome or Selenium install is needed**.
It also fixes two problems of the old notebook: indices queried by display name came back **empty**, and the
`high`, `low`, `ltP` and `ptsC30d` columns were always `-`. The old notebook is kept next to this one as
`NSE_ALL_INDEX_downloader.selenium-backup.ipynb`.

**Before you start**, put these 3 files in `MyDrive/screener-document-download/code/` (the folder the Screener notebooks use):
`nse_indices.py`, `config.py`, `log_setup.py`.

Run the cells in order: **1** connect Drive, **2** settings, **3** quick test, **4** real download, **5** check the files, **6** sync Drive.
Safe to re-run: indices already saved today are skipped (set `FORCE = True` in Cell 2 to download them again).

In [ ]:
# Cell 1 - connect Google Drive and check the script files
import os
from google.colab import drive

drive.mount('/content/drive')   # if you get "Transport endpoint is not connected": drive.mount('/content/drive', force_remount=True)

PROJECT = '/content/drive/MyDrive/screener-document-download'   # folder that holds code/
CODE_DIR = f'{PROJECT}/code'
missing = [f for f in ('nse_indices.py', 'config.py', 'log_setup.py') if not os.path.exists(f'{CODE_DIR}/{f}')]
if missing:
    raise FileNotFoundError(f'Missing in {CODE_DIR}: {missing}. Upload them to Drive and run this cell again.')
print('OK - script files found in', CODE_DIR)

In [ ]:
# Cell 2 - settings (edit, then run the cell)
OUT_DIR = '/content/drive/MyDrive/NSEDATA-ALL-INDEX'   # files go to OUT_DIR/<YYYYMMDD>/
LOG_DIR = f'{PROJECT}/logs'
FORCE = False    # True = download indices already saved today again

force_arg = '--force' if FORCE else ''
os.makedirs(OUT_DIR, exist_ok=True)
print('Output folder:', OUT_DIR)
print('Force re-download:', FORCE)

In [ ]:
# Cell 3 - QUICK TEST: fetches the index list and NIFTY 50 only, saves nothing.
# If this fails, NSE may be blocking Colab's servers; see the message it prints.
import sys, importlib

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import nse_indices as nse
importlib.reload(nse)

try:
    session = nse.make_session()
    targets, rows = nse.list_indices(session)
    print(f'Index list OK: {len(targets)} indices, e.g. {targets[:3]}')
    items = nse.fetch_index(session, 'NIFTY 50')
    print(f'NIFTY 50 OK: {len(items)} rows (expect about 51). First row:', items[0].get('symbol'), items[0].get('lastPrice'))
except Exception as exc:
    print('TEST FAILED:', exc)
    print('If it says HTTP 403 / 401 / not JSON, NSE is blocking this Colab session. Try again later or run the script on your own computer.')

In [ ]:
# Cell 4 - REAL DOWNLOAD of all indices. Takes about 4-5 minutes. Progress prints below; the log goes to LOG_DIR.
!python3 {CODE_DIR}/nse_indices.py --out {OUT_DIR} --log-dir {LOG_DIR} {force_arg}

In [ ]:
# Cell 5 - CHECK what was saved (latest date folder)
import glob, time
import pandas as pd

days = sorted(d for d in os.listdir(OUT_DIR) if d.isdigit() and len(d) == 8 and os.path.isdir(f'{OUT_DIR}/{d}'))
if not days:
    print('No dated folder found yet - run Cell 4 first.')
else:
    day_dir = f'{OUT_DIR}/{days[-1]}'
    for _ in range(5):                       # Drive can take a moment to show new files
        index_files = [f for f in glob.glob(f'{day_dir}/*.csv') if not os.path.basename(f).startswith('NSE_ALL_INDICES_')]
        if index_files:
            break
        time.sleep(1)
    masters = glob.glob(f'{day_dir}/NSE_ALL_INDICES_*.csv')
    header_only = [os.path.basename(f) for f in index_files if sum(1 for _ in open(f, encoding='utf-8')) <= 1]
    print(f'Folder: {day_dir}')
    print(f'Index files: {len(index_files)} | master list files: {len(masters)} | header-only (no rows): {len(header_only)}')
    if header_only:
        print('Header-only files (re-run Cell 4 to retry):', header_only[:10])
    nifty = f'{day_dir}/NIFTY_50.csv'
    if os.path.exists(nifty):
        print('\nNIFTY_50.csv:')
        display(pd.read_csv(nifty).head(5))

In [ ]:
# Cell 6 - when finished: make sure everything is written to Drive
drive.flush_and_unmount()
print('Drive synced.')